# 10 Transformer Encoder 整体结构

前面我们已经把进入 Transformer Encoder 需要的零件都学过了：

$$
\begin{aligned}
&\operatorname{TokenEmbedding}\\
&\text{位置编码}\\
&\operatorname{MultiHeadSelfAttention}\\
&\operatorname{FeedForwardNetwork}\\
&\text{残差连接}\\
&\operatorname{LayerNorm}
\end{aligned}
$$

这一节不再继续拆零件。

这一节做一件事：

$$
\text{已学组件}\rightarrow\text{完整的 Transformer Encoder Layer}
$$

学完这一节，你应该能说清楚一条输入是怎么流过 Encoder 的。

## 1. Encoder 是什么

Encoder 的中文意思是编码器。

它的作用可以先理解成：

$$
\text{输入序列}\xrightarrow{\operatorname{Encoder}}\text{带上下文信息的向量表示}
$$

例如输入一句话：

$$
\text{小红}\quad\text{明天}\quad\text{要}\quad\text{考试}
$$

一开始，每个 token 只有比较基础的 embedding。

经过 Encoder 后，每个 token 的表示都会融合上下文。

也就是说，Encoder 的输出仍然是一串向量。

但这些向量已经比原始 embedding 更懂这句话。

## 2. Encoder 和 Decoder 先不要混

Transformer 完整结构里通常有 Encoder 和 Decoder。

但这一节只学习 Encoder。

Encoder 的重点是：

$$
\operatorname{Encoder}:\text{完整输入序列}\rightarrow\text{每个位置的上下文表示}
$$

Decoder 的重点是：

$$
\operatorname{Decoder}:\text{已有输出}\rightarrow\text{逐步生成输出序列}
$$

以后如果学机器翻译或 GPT，再专门讲 Decoder。

现在先把 Encoder 这半边学清楚。

## 3. Encoder 的输入是什么

假设输入是一批序列。

经过 token embedding 后，形状是：

$$
X:B\times N\times D
$$

其中：

- $B$ 表示 batch size。
- $N$ 表示 token 数量。
- $D$ 表示每个 token 的向量维度，也叫模型维度。

然后加入位置编码：

$$
Z=X+P
$$

加入位置编码后，形状仍然是：

$$
Z:B\times N\times D
$$

这个 $Z$ 就是送进 Encoder Layer 的输入。

## 4. 一个 Encoder Layer 由哪些部分组成

一个经典 Transformer Encoder Layer 可以先看成两大段：

$$
\begin{aligned}
\text{第一段}&:\operatorname{MultiHeadSelfAttention}+\operatorname{AddNorm}\\
\text{第二段}&:\operatorname{FeedForwardNetwork}+\operatorname{AddNorm}
\end{aligned}
$$

写成流程：

$$
X\rightarrow\operatorname{MultiHeadSelfAttention}\rightarrow\operatorname{AddNorm}\rightarrow\operatorname{FeedForwardNetwork}\rightarrow\operatorname{AddNorm}\rightarrow O
$$

这就是一层 Encoder Layer 的骨架。

你前面学的每个零件，都会在这里出现。

## 5. 第一段：Multi-Head Self-Attention 做什么

输入是：

$$
X:B\times N\times D
$$

Multi-Head Self-Attention 的作用是让 token 之间交流。

它会让每个 token 用多个 head 去看同一句话里的其他 token。

输出形状通常仍然是：

$$
\operatorname{MHA}(X):B\times N\times D
$$

注意：形状没变，但信息变了。

原来每个 token 主要是自己的表示。

经过 MHA 后，每个 token 融合了上下文信息。

## 6. 第一段：Add & Norm 怎么接上

Multi-Head Attention 输出后，不是直接进入 FFN。

还要先做 Add & Norm。

经典 Post-Norm 写法可以写成：

$$
H=\operatorname{LayerNorm}(X+\operatorname{MHA}(X))
$$

这里：

$$
\begin{aligned}
X&:\text{原输入}\\
\operatorname{MHA}(X)&:\text{Attention 产生的新信息}\\
X+\operatorname{MHA}(X)&:\text{残差连接}\\
\operatorname{LayerNorm}&:\text{稳定数值分布}
\end{aligned}
$$

形状路线是：

$$
\begin{aligned}
(B\times N\times D)+(B\times N\times D)&\rightarrow B\times N\times D\\
\operatorname{LayerNorm}(B\times N\times D)&\rightarrow B\times N\times D
\end{aligned}
$$

所以 $H$ 的形状仍然是：

$$
H:B\times N\times D
$$

## 7. 第二段：FFN 做什么

第一段输出 $H$ 后，进入 FFN。

FFN 的作用是对每个 token 单独做非线性加工。

它不负责 token 之间交流。

token 之间交流已经由前面的 Multi-Head Attention 完成。

FFN 的常见形状路线是：

$$
H:B\times N\times D\rightarrow B\times N\times d_{\mathrm{ff}}\rightarrow B\times N\times D
$$

所以：

$$
\operatorname{FFN}(H):B\times N\times D
$$

它输出和输入形状一致，方便后面再次做残差连接。

## 8. 第二段：再次 Add & Norm

FFN 后面也要做 Add & Norm。

经典 Post-Norm 写法是：

$$
O=\operatorname{LayerNorm}(H+\operatorname{FFN}(H))
$$

这里：

$$
\begin{aligned}
H&:\text{FFN 前的输入}\\
\operatorname{FFN}(H)&:\text{FFN 加工出的新信息}\\
H+\operatorname{FFN}(H)&:\text{残差连接}\\
\operatorname{LayerNorm}&:\text{稳定数值分布}
\end{aligned}
$$

形状路线仍然是：

$$
\begin{aligned}
(B\times N\times D)+(B\times N\times D)&\rightarrow B\times N\times D\\
\operatorname{LayerNorm}(B\times N\times D)&\rightarrow B\times N\times D
\end{aligned}
$$

$O$ 就是这一层 Encoder Layer 的输出。

## 9. 把一层 Encoder Layer 串起来

现在把一层完整串起来：

$$
\begin{aligned}
X&:B\times N\times D\\
A&=\operatorname{MHA}(X),\quad A:B\times N\times D\\
H&=\operatorname{LayerNorm}(X+A),\quad H:B\times N\times D\\
F&=\operatorname{FFN}(H),\quad F:B\times N\times D\\
O&=\operatorname{LayerNorm}(H+F),\quad O:B\times N\times D
\end{aligned}
$$

这就是一层 Transformer Encoder Layer 的完整主线。

它的输入和输出形状一样。

但输出里的每个 token 表示已经被进一步更新。

## 10. 为什么 Encoder Layer 输入输出形状保持一致

一个 Encoder Layer 输入是：

$$
B\times N\times D
$$

输出也是：

$$
B\times N\times D
$$

这有几个好处。

第一，方便残差连接。

因为残差连接要求形状一致。

第二，方便堆叠多层 Encoder。

如果第 1 层输出还是 $B\times N\times D$，就可以直接送进第 2 层。

第三，方便保持每个 token 的位置结构。

Encoder 不会把序列压成一个向量。

它仍然保留 $N$ 个 token 位置，只是每个位置的表示变得更有上下文。

## 11. 堆叠多个 Encoder Layers

实际 Transformer 通常会堆很多个 Encoder Layer。

例如堆 6 层：

$$
\operatorname{EncoderLayer}_1\rightarrow\operatorname{EncoderLayer}_2\rightarrow\operatorname{EncoderLayer}_3\rightarrow\operatorname{EncoderLayer}_4\rightarrow\operatorname{EncoderLayer}_5\rightarrow\operatorname{EncoderLayer}_6
$$

每一层输入输出形状都可以是：

$$
B\times N\times D
$$

但每一层都会进一步更新 token 表示。

可以先这样理解：

$$
\begin{aligned}
\text{浅层}&:\text{学习较直接、较简单的关系}\\
\text{深层}&:\text{基于已有表示学习更抽象的关系}
\end{aligned}
$$

这和我们前面 3Blue1Brown 复盘课里说的“层层改写 embedding”是同一件事。

## 12. Encoder 输出是什么

经过多层 Encoder 后，输出仍然是一组 token 表示：

$$
B\times N\times D
$$

这和 CNN 里最后得到一组特征图有点类似。

Encoder 输出可以理解成：

$$
\operatorname{EncoderOutput}=\left\{\text{每个 token 的上下文表示}\right\}
$$

例如句子中“她”这个 token，经过 Encoder 后，它的向量可能已经融合了“小红”“考试”“明天”等信息。

所以 Encoder 输出不是最终答案。

它是一组更高级的特征表示。

后面可以接不同任务头完成不同任务。

## 13. Encoder 输出可以怎么用于任务

Encoder 输出是：

$$
B\times N\times D
$$

不同任务会使用这些输出的不同部分。

例如序列分类任务：

$$
\text{句子表示}\xrightarrow{\text{分类头}}\text{正面或负面}
$$

可以取某个特殊 token 的输出，接分类头。

例如 token 级别任务：

$$
\text{每个 token 的表示}\xrightarrow{\text{分类头}}\left\{\text{人名、地点、组织等标签}\right\}
$$

可以对每个 token 的输出分别接分类头。

例如机器翻译中的 Encoder-Decoder 结构：

$$
\operatorname{EncoderOutput}\xrightarrow{\text{上下文信息}}\operatorname{Decoder}
$$

这一节先不展开任务头，只先知道 Encoder 输出可以被后续模块使用。

## 14. Encoder 和 BERT 的关系

BERT 是一种典型的 Encoder-only Transformer。

也就是说，它主要由多层 Transformer Encoder 堆叠而成。

BERT 的特点是适合做理解类任务，比如：

$$
\operatorname{BERT}\rightarrow\left\{\begin{array}{l}\text{文本分类}\\\text{命名实体识别}\\\text{句子匹配}\\\text{阅读理解}\end{array}\right.
$$

为什么适合理解类任务？

因为 Encoder 可以让每个 token 双向参考整段输入。

它不需要像 GPT 那样严格只能看前文。

不过 BERT 的训练目标和具体结构以后再讲。

这里先建立关系：

$$
\operatorname{BERT}=\text{多层 Transformer Encoder 的重要代表}
$$

## 15. CLS Token 是什么

前面第 13 节提到“取某个特殊 token 的输出接分类头”，这个特殊 token 通常就是 CLS token。

CLS 是 classification 的缩写，一般记作 [CLS]。

在 BERT 等模型里，它被加在输入序列的最开头：

$$
\text{[CLS]}\quad\text{小红}\quad\text{明天}\quad\text{要}\quad\text{考试}
$$

它本身不是一个真实的词，而是一个专门用来存放“整句信息”的位置。

为什么需要它？

因为 Self-Attention 会让所有 token 互相加权，但并没有一个天然用来存放“整句表示”的固定位置。

[CLS] 经过多层 Encoder 后，会和序列里的所有 token 交互，于是它的向量就聚合了整句的语义。

做句子分类时，只取 [CLS] 对应的输出向量接一个分类头即可：

$$
\text{[CLS] 的输出}\xrightarrow{\text{分类头}}\text{正面 / 负面}
$$

对应的还有 [SEP]，用于分隔两个句子或标记句子结尾。

入门先记住一句核心：

$$
\text{CLS token：放在句首，用于汇总整句信息，常被用来做分类}
$$


## 16. Encoder 和 GPT 的区别先简单了解

GPT 通常是 Decoder-only Transformer。

它更强调根据前文生成后文。

所以 GPT 中会使用 causal mask，防止当前位置看到未来 token。

Encoder 通常用于理解输入。

在没有特殊限制时，Encoder 的 Self-Attention 可以让每个位置看到整段输入。

简单对比：

$$
\begin{aligned}
\operatorname{Encoder}&:\text{编码整段输入，常用于理解任务}\\
\operatorname{Decoder}&:\text{逐步生成输出，常用于生成任务}
\end{aligned}
$$

这只是入门级区分。

后面如果学 GPT 或完整 Transformer Decoder，再展开更多细节。

## 17. 用一个小例子完整走一遍形状

假设：

$$
\begin{aligned}
B&=2,&N&=4,&D&=8,\\
h&=2,&d_{\mathrm{head}}&=4,&d_{\mathrm{ff}}&=32
\end{aligned}
$$

输入 token embedding：

$$
X:2\times4\times8
$$

加位置编码：

$$
Z:2\times4\times8
$$

Multi-Head Attention：

$$
\begin{aligned}
Q,K,V&:2\times2\times4\times4\\
\text{注意力表}&:2\times2\times4\times4\\
\operatorname{MHA}(X)&:2\times4\times8
\end{aligned}
$$

第一次 Add & Norm：

$$
2\times4\times8
$$

FFN：

$$
2\times4\times8\rightarrow2\times4\times32\rightarrow2\times4\times8
$$

第二次 Add & Norm：

$$
2\times4\times8
$$

这一层 Encoder 输出：

$$
2\times4\times8
$$

## 18. 为什么说 Encoder 是组装课

现在再回头看，Encoder 其实没有太多凭空冒出来的新概念。

它是把前面学过的模块按固定顺序组合起来。

对应关系如下：

$$
\begin{aligned}
\operatorname{TokenEmbedding}&:\text{把 token 变成向量}\\
\text{位置编码}&:\text{加入顺序信息}\\
\operatorname{MultiHeadAttention}&:\text{让 token 之间交流}\\
\text{残差连接}&:\text{保留原信息，学习增量}\\
\operatorname{LayerNorm}&:\text{稳定数值分布}\\
\operatorname{FFN}&:\text{对每个 token 单独做非线性加工}\\
\text{第二次残差连接}+\operatorname{LayerNorm}&:\text{合并并稳定 FFN 后的结果}
\end{aligned}
$$

所以这一节最重要的不是背结构图。

而是知道每个模块为什么放在那里。

## 19. 常见误解 1：Encoder 输出一个向量

不一定。

标准 Encoder 的输出通常仍然是一组 token 表示：

$$
B\times N\times D
$$

也就是说，每个 token 都有自己的输出向量。

如果某个任务需要一个整体句子向量，可以后续再做池化、取特殊 token，或者接任务头。

但 Encoder 本身不是一上来就把整个序列压成一个向量。

## 20. 常见误解 2：Attention 和 FFN 做的是同一件事

不是。

它们分工不同。

Attention：

$$
\operatorname{Attention}:\text{不同 token 之间交换信息}
$$

FFN：

$$
\operatorname{FFN}:\text{对每个 token 的当前表示做非线性加工}
$$

所以 Encoder Layer 是先交流，再加工。

这两个模块互补，不是重复。

## 21. 常见误解 3：Add & Norm 可有可无

Add & Norm 不是装饰。

残差连接帮助深层网络传递信息和梯度。

LayerNorm 帮助稳定数值分布。

如果没有这些结构，Transformer 堆很多层会更难训练。

所以在理解 Encoder 时，不要只盯着 Attention。

真正的 Encoder Layer 是多个结构配合工作。

## 22. 本节小结

这一节先记住：

1. Encoder 的作用是把输入序列编码成上下文表示。
2. Encoder 输入通常是加了位置编码的 token embedding，形状是 $B\times N\times D$。
3. 一个 Encoder Layer 包含两大段：MHA + Add & Norm，FFN + Add & Norm。
4. MHA 负责 token 之间交流。
5. FFN 负责每个 token 内部非线性加工。
6. 残差连接保留原信息，LayerNorm 稳定数值分布。
7. Encoder Layer 输入输出形状通常保持 $B\times N\times D$。
8. 多层 Encoder 可以连续堆叠，让 token 表示逐层变得更有上下文。
9. Encoder 输出不是最终答案，而是一组高级特征表示。
10. BERT 是 Encoder-only Transformer 的重要代表。11. CLS token 是放在句首用于汇总整句信息的特殊 token，常被用来做分类。

## 23. 自测问题

1. Transformer Encoder 的主要作用是什么？
2. Encoder 和 Decoder 的基本区别是什么？
3. Encoder 输入为什么要加位置编码？
4. 一个 Encoder Layer 通常由哪两大段组成？
5. MHA 在 Encoder Layer 里负责什么？
6. FFN 在 Encoder Layer 里负责什么？
7. 为什么 MHA 后面要接 Add & Norm？
8. 为什么 FFN 后面也要接 Add & Norm？
9. 为什么 Encoder Layer 输入输出形状通常都是 $B\times N\times D$？
10. 多层 Encoder 堆叠时，每一层的输出表示有什么变化？
11. Encoder 输出通常是什么形状？
12. 为什么说 Encoder 输出不是最终答案？
13. BERT 和 Transformer Encoder 有什么关系？
14. 为什么不能把 Add & Norm 看成可有可无的装饰？15. CLS token 是什么？它通常放在序列的什么位置？有什么作用？